In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.1 Residual stream

A transformer block does not replace its input, it *adds* to it: `x + f(x)`.
That running sum is the **residual stream**. Gradients flow straight through
the `+`, so even deep stacks keep training instead of vanishing.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
x = torch.randn(1, 4, dim)
sublayer = nn.Linear(dim, dim)
delta = sublayer(x)
residual_out = x + delta   # the residual connection
print("output = input + a learned delta")

Each block reads the stream, computes a delta, and writes it back. Information
from layer 1 can survive untouched all the way to the output.

In [ ]:
# (x + delta) - x recovers delta up to floating-point cancellation.
assert torch.allclose(residual_out - x, delta, atol=1e-5)